## Installations & Label Definitons

In [1]:
# Cell 1: Install & import libs (FIXED)
import sys
import subprocess
import os

def install_package(package):
    """Safely install a package using pip"""
    try:
        # Try using pip directly
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ Successfully installed {package}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"❌ Failed to install {package}: {e}")
        return False

def check_package(package_name, import_name=None):
    """Check if a package is already installed"""
    if import_name is None:
        import_name = package_name
    try:
        __import__(import_name)
        print(f"✅ {package_name} is already installed")
        return True
    except ImportError:
        print(f"⚠️  {package_name} not found, installing...")
        return False

print("🔧 Checking and installing required packages...")

packages = [
    ("pandas", "pandas"),
    ("openai", "openai"),
    ("scikit-learn", "sklearn")
]

for package, import_name in packages:
    if not check_package(package, import_name):
        install_package(package)

print("\n📦 Importing libraries...")
try:
    import pandas as pd
    import openai
    print("✅ All imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("💡 Try restarting the kernel and running this cell again")

print("\n🎯 Ready to proceed!")

🔧 Checking and installing required packages...
✅ pandas is already installed
✅ openai is already installed
✅ scikit-learn is already installed

📦 Importing libraries...
✅ All imports successful!

🎯 Ready to proceed!


In [2]:
# Cell 2 : System diagnostic and package version checker
import sys
import platform
import os

print("🔍 SYSTEM DIAGNOSTICS")
print("=" * 40)
print(f"Python version: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Architecture: {platform.architecture()}")
print(f"Python executable: {sys.executable}")
print(f"Current working directory: {os.getcwd()}")

print("\n📦 PACKAGE VERSIONS:")
packages_to_check = {
    'pandas': 'pd',
    'openai': 'openai', 
    'sklearn': 'sklearn',
    'numpy': 'np'
}

for package, alias in packages_to_check.items():
    try:
        if package == 'pandas':
            import pandas as pd
            print(f"✅ pandas: {pd.__version__}")
        elif package == 'openai':
            import openai
            print(f"✅ openai: {openai.__version__}")
        elif package == 'sklearn':
            import sklearn
            print(f"✅ scikit-learn: {sklearn.__version__}")
        elif package == 'numpy':
            import numpy as np
            print(f"✅ numpy: {np.__version__}")
    except ImportError:
        print(f"❌ {package}: Not installed")
    except Exception as e:
        print(f"⚠️  {package}: Error - {e}")

print("\n🎯 All systems ready for multi-label classification!")

# List of labels
labels = ["Medical", "Mental Health", "Abuse", "Aggression", "Sexual", "Discrimination", "Pregnancy", "Not Applicable"]

# Updated definitions (unchanged for now)
definitions = {
    "Medical": "The post should be assigned this label if it talks about taking medical pills, going through procedures or treatments related to pregnancy loss (like abortion or miscarriage), or getting medical care after sexual violence. It also includes mentions of bleeding, blood, or anything graphic like gore related to pregnancy loss or sexual violence.",
    
    "Mental Health": "The post should be assigned this label if it talks about emotional pain like sadness, crying, anxiety, panic, or stress.",
    
    "Abuse": "The post should be assigned this label if someone is being controlled, threatened, or harmed emotionally or physically by another person. This includes situations where a person is forced to do something, made to feel unsafe, or has their freedom or privacy violated. It includes coercion, manipulation, intimidation, stalking, voyeurism, rape, hate speech, and all forms of sexual violence.",
    
    "Aggression": "The post should be assigned this label if it describes someone being physically attacked, threatened, or facing violent behavior during or after a traumatic event. It includes things like hitting, pushing, chasing, restraining, or threats of harm.",
    
    "Sexual": "The post should be assigned this label if it includes sexual content such as sex acts, sexual behavior, pornography, kinks, or sexual preferences, whether harmful or not. This also includes kinks that do not involve sex acts, like praise kink or size kink — they still count as sexual content.",
    
    "Discrimination": "The post should be assigned this label when someone is judged, mistreated, or shamed because of their gender, body, identity, or background.",
    
    "Pregnancy": "The post should be assigned this label if it talks about pregnancy loss, like miscarriage, abortion, or stillbirth, or complications related to pregnancy. It should not be used for general pregnancy or childbirth unless it is clearly connected to a loss or medical issue.",
    
    "Not Applicable": "Used when the post doesn't match any of the above trigger categories or topics."
}


🔍 SYSTEM DIAGNOSTICS
Python version: 3.10.12 (main, May 27 2025, 17:12:29) [GCC 11.4.0]
Platform: Linux-6.8.0-1035-aws-x86_64-with-glibc2.35
Architecture: ('64bit', 'ELF')
Python executable: /usr/bin/python3
Current working directory: /home/ubuntu

📦 PACKAGE VERSIONS:
✅ pandas: 2.3.1
✅ openai: 1.99.1
✅ scikit-learn: 1.7.1
✅ numpy: 2.2.6

🎯 All systems ready for multi-label classification!


## Dataset

In [3]:
# Cell 3 : Load Dataset
df = pd.read_csv("Combined_Dataset_Annotations - Combined_Dataset.csv")
df.head()

,id,soure,subreddit,title,body,created_utc,url,Tags
0,1ljxynj,abortion,abortion,Complications after abortion?,"Hi everyone, Ive read that abortions don’t cau...",2025-06-25 5:59:34,https://www.reddit.com/r/abortion/comments/1lj...,"Medical, Sexual, Pregnancy"
1,1ljxtt8,NaN,abortion,Second MA abortion today and I'm absolutely te...,I'm having my second MA abortion today and I'm...,2025-06-25 5:51:08,https://www.reddit.com/r/abortion/comments/1lj...,"Medical, Pregnancy, Mental Health"
2,1ljwhkb,abortion,abortion,Help needed/ live in Texas where abortion in b...,Anyone know of a legit site to support women i...,2025-06-25 4:31:43,https://www.reddit.com/r/abortion/comments/1lj...,"Discrimination, Pregnancy"
3,1ljvy6u,NaN,abortion,medical abortion at 6 weeks,I’ll be doing my procedure on Friday and I got...,2025-06-25 4:02:28,https://www.reddit.com/r/abortion/comments/1lj...,"Medical, Pregnancy"
4,1ljv5k4,abortion,abortion,Idk what to feel about my decision after doing...,I just had medical abortion yesterday. I was a...,2025-06-25 3:19:54,https://www.reddit.com/r/abortion/comments/1lj...,"Pregnancy, Mental Health"


In [4]:
# Cell 4 : Step 1: Clean 'Tags' column if needed
df["Tags"] = df["Tags"].apply(
    lambda x: [tag.strip() for tag in x.split(",")] if isinstance(x, str) else x
)

# Step 2: Create binary columns per label
all_labels = [
    "Medical",
    "Mental Health",
    "Abuse",
    "Aggression",
    "Sexual",
    "Discrimination",
    "Pregnancy",
    "Not Applicable"
]

for label in all_labels:
    df[label] = df["Tags"].apply(lambda tags: int(label in tags if isinstance(tags, list) else []))


## Prompts

In [22]:
# # # Cell 5 : Prompt 1

# def make_per_label_prompt(title, body, label_name, definition):
#     """
#     Create a chain-of-thought prompt for evaluating a single label.

#     Arguments:
#     - title: Reddit post title
#     - body: Reddit post description
#     - label_name: The label we are testing (e.g., "Medical", "Abuse")
#     - definition: The definition text for this label

#     Returns:
#     - Formatted prompt string for the LLM
#     """

#     prompt = (
#     "You are an attentive content reviewer tasked with identifying specific emotional or traumatic triggers "
#     "in Reddit posts related to abortion, miscarriage, or sexual violence.\n\n"
#     "You will evaluate **only one trigger label at a time** using a step by step approach.\n\n"
#     f"Trigger Warning Label to evaluate: **{label_name}**\n\n"
#     f"Definition of '{label_name}':\n"
#     f"{definition.strip()}\n\n"
#     "Step-by-step instructions:\n"
#     "1. Carefully read the post (Title and Description).\n"
#     "2. Think about whether the post **clearly matches** the above definition.\n"
#     "3. Be cautious — if the post is vague or not directly related to the label, answer NO.\n"
#     "4. If there is strong evidence that matches the definition, answer YES.\n"
#     "5. Do not assume or guess based on weak signals.\n\n"
#     f"Title: {title.strip()}\n\n"
#     f"Description: {body.strip()}\n\n"
#     "Answer with only YES or NO.\n"
#     f"Does this post contain the '{label_name}' trigger?\n"
#     "Answer:"
# )

#     return prompt


In [23]:
# Cell 6 : Prompt 2

def make_per_label_prompt(title, body, label_name, definition):
    """
    Create a chain-of-thought prompt for evaluating a single label.

    Arguments:
    - title: Reddit post title
    - body: Reddit post description
    - label_name: The label we are testing (e.g., "Medical", "Abuse")
    - definition: The definition text for this label

    Returns:
    - Formatted prompt string for the LLM
    """

#     prompt = (
#     "You are an attentive content reviewer tasked with identifying specific emotional or traumatic triggers "
#     "in Reddit posts related to abortion, miscarriage, or sexual violence.\n\n"
#     "You should analyze each post and evaluate whether readers could be triggered by it. "
#     "We have multiple categories of trigger warnings that you can provide, and in this case, "
#     "You will evaluate **only one trigger label at a time** using a step by step approach.\n\n"
#     f"Trigger Warning Label to evaluate: *{label_name}*\n\n"
#     f"Definition of '{label_name}':\n"
#     f"{definition.strip()}\n\n"
#     "Step-by-step instructions:\n"
#     "1. Carefully read the post (Title and Description).\n"
#     "2. Think step by step about whether the post *clearly matches* the above definition.\n"
#     "3. Be cautious — if the post is vague or not directly related to the label, answer NO.\n"
#     "4. If there is strong evidence that matches the definition, answer YES.\n"
#     "5. Do not assume or guess based on weak signals.\n\n"
#     f"Title: {title.strip()}\n\n"
#     f"Description: {body.strip()}\n\n"
#     "Answer with only YES or NO.\n"
#     f"Does this post contain the '{label_name}' trigger?\n"
#     "Answer:"
# )

#     return prompt

    prompt = (
    "You are an attentive content reviewer tasked with identifying specific emotional or traumatic triggers "
    "in Reddit posts related to abortion, miscarriage, or sexual violence.\n\n"
    "Your task is to analyze the post and decide whether readers could be triggered by it. "
    "We have multiple categories of trigger warnings, but in this case, "
    "you will evaluate *only one trigger label at a time* using a careful step-by-step approach.\n\n"
    f"Trigger Warning Label to evaluate: {label_name}\n\n"
    f"Definition of '{label_name}':\n"
    f"{definition.strip()}\n\n"
    "Global constraints:\n"
    "• Answer strictly with YES or NO (uppercase, no punctuation, no explanation).\n"
    "• Only mark a label YES if the post explicitly and clearly matches the label’s definition.\n"
    "• If the post is vague, uncertain, or does not directly match the definition, answer NO.\n"
    "• Do not infer, assume, or guess based on weak signals.\n\n"
    "Special rule for 'Not Applicable' (NA):\n"
    "• 'Not Applicable' must be marked YES *only if no other trigger labels would be YES* for this post.\n"
    "• If any other trigger label (evaluated separately) would be YES, then 'Not Applicable' must be NO.\n"
    "• In other words, 'Not Applicable' is mutually exclusive with all other labels.\n\n"
    "Step-by-step instructions:\n"
    "1) Carefully read the post (Title and Description).\n"
    "2) Compare the content to the label’s definition above.\n"
    "3) Decide: Does the post explicitly and clearly match this label? If yes, answer YES; otherwise, NO.\n"
    "4) If the current label is 'Not Applicable', apply the special rule above.\n\n"
    f"Title: {title.strip()}\n\n"
    f"Description: {body.strip()}\n\n"
    f"Does this post contain the '{label_name}' trigger?\n"
    "Answer:"
    )

    return prompt


In [27]:
# #Prompt 3

# def make_per_label_prompt(title, body, label_name, definition):
#     """
#     Create a chain-of-thought prompt for evaluating a single label.

#     Arguments:
#     - title: Reddit post title
#     - body: Reddit post description
#     - label_name: The label we are testing (e.g., "Medical", "Abuse")
#     - definition: The definition text for this label

#     Returns:
#     - Formatted prompt string for the LLM
#     """

#     prompt = (
#     "You are an attentive content reviewer tasked with identifying specific emotional or traumatic triggers "
#     "in Reddit posts related to abortion, miscarriage, or sexual violence.\n\n"
#     "Your task is to analyze the post and decide whether readers could be triggered by it. "
#     "We have multiple categories of trigger warnings, but in this case, "
#     "you will evaluate **only one trigger label at a time** using a careful step-by-step approach.\n\n"
#     f"Trigger Warning Label to evaluate: *{label_name}*\n\n"
#     f"Definition of '{label_name}':\n"
#     f"{definition.strip()}\n\n"
#     "Step-by-step instructions:\n"
#     "1. Carefully read the post (Title and Description).\n"
#     "2. Decide whether the post *explicitly and clearly matches* the above definition.\n"
#     "3. If the post is vague, uncertain, or not directly related to the label, answer NO.\n"
#     "4. If there is strong, explicit evidence that matches the definition, answer YES.\n"
#     "5. Do not infer, assume, or guess based on weak signals.\n\n"
#     f"Title: {title.strip()}\n\n"
#     f"Description: {body.strip()}\n\n"
#     "Answer with only YES or NO (uppercase, no punctuation, no explanation).\n"
#     f"Does this post contain the '{label_name}' trigger?\n"
#     "Answer:"
# )


#     return prompt



In [6]:
# #prompt 4
# def make_per_label_prompt(title, body, label_name, definition):
#     """
#     Create a chain-of-thought prompt for evaluating a single label.

#     Arguments:
#     - title: Reddit post title
#     - body: Reddit post description
#     - label_name: The label we are testing (e.g., "Medical", "Abuse")
#     - definition: The definition text for this label

#     Returns:
#     - Formatted prompt string for the LLM
#     """

#     prompt = (
#     "You are an attentive content reviewer tasked with identifying specific emotional or traumatic triggers "
#     "in Reddit posts related to abortion, miscarriage, or sexual violence.\n\n"
#     "Your task is to analyze the post and decide whether readers could be triggered by it. "
#     "We have multiple categories of trigger warnings, but in this case, "
#     "you will evaluate **only one trigger label at a time** using a careful step-by-step approach.\n\n"
#     f"Trigger Warning Label to evaluate: *{label_name}*\n\n"
#     f"Definition of '{label_name}':\n"
#     f"{definition.strip()}\n\n"
#     "Step-by-step instructions:\n"
#     "1. Carefully read the post (Title and Description).\n"
#     "2. Decide whether the post *explicitly and directly matches* the above definition.\n"
#     "3. Only answer YES if the post literally and directly matches the definition. "
#     "Do not label based on implications, assumptions, or indirect context.\n"
#     "4. If the post is vague, uncertain, or does not strongly fit the label, the correct answer is NO "
#     "(so that it may be handled as 'Not Applicable').\n"
#     "5. Remember: this is a strict binary choice — either YES (clear evidence) or NO (everything else).\n\n"
#     f"Title: {title.strip()}\n\n"
#     f"Description: {body.strip()}\n\n"
#     "Answer with only one token: YES or NO (uppercase, no punctuation, no explanation).\n"
#     f"Does this post contain the '{label_name}' trigger?\n"
#     "Answer:"
# )



#     return prompt

In [24]:
# # Cell 7 

# from sentence_transformers import SentenceTransformer
# import numpy as np
# from sklearn.metrics.pairwise import cosine_similarity
# import torch
# import json
# import pandas as pd
# global collected_examples_per_label
# collected_examples_per_label = {}  # ✅ Reset every time you call the function


# # Assumes labels and definitions are already defined
# df = pd.read_csv("annotations_dataset_with_index.csv")
# embeddings = torch.load("post_embeddings.pt")
# model = SentenceTransformer("all-mpnet-base-v2")
# with open("similar_posts_k5.json", "r") as f:
#     similar_post_map = json.load(f)

# def make_multishot_prompt_cosine(k, title, body, current_index=None, include_defs=True, debug=False):
#     global collected_examples_per_label
#     collected_examples_per_label = {}  # Reset on each new call

#     input_text = title + " " + body
#     input_embedding = model.encode([input_text])[0]
    
#     # instr = (
#     #     "You are an attentive content reviewer tasked with identifying potential emotional or traumatic triggers "
#     #     "in Reddit posts related to abortion, miscarriage, or sexual violence.\n\n"
#     #     "You should analyze each post and evaluate whether readers could be triggered by it. "
#     #     "We have multiple categories of trigger warnings that you can provide, and you will assess each label one at a time "
#     #     "using examples from similar posts and by thinking step by step.\n\n"
#     # )

#     # instr = (
#     #     "You are a attentive content reviewer tasked with identifying specific emotional or traumatic triggers "
#     #     "in Reddit posts related to abortion, miscarriage, or sexual violence.\n\n"
#     #     "You will evaluate **only one trigger label at a time** using examples from similar posts and step by step approach.\n\n"
#     # )

#     instr = (
#     "You are a attentive content reviewer for Reddit posts. "
#     "Your task is to identify specific emotional or traumatic triggers, "
#     "but you will only consider triggers when the posts are explicitly related to "
#     "abortion, miscarriage, or sexual violence.\n\n"
#     "You will evaluate **one label at a time** using definitions and similar examples.\n\n"
#     "General decision rules:\n"
#     "• Mark **YES** only when the post has clear, explicit evidence for that label **and** it is connected to abortion, miscarriage, or sexual violence.\n"
#     "• If the connection is unclear, indirect, or unrelated to these topics, mark **NO**.\n"
#     "• Avoid inferring from generic symptoms, emotions, or events that are not explicitly tied to the core topics.\n"
#     "• Prefer precision over recall — if in doubt, answer NO.\n"
#     "• If none of the labels apply, mark **Not Applicable**.\n"
# )


#     defs_text = ""
#     if include_defs:
#         defs_text = "Trigger Warning Definitions:\n" + "\n".join(
#             f"• {l}: {v.strip()}" for l, v in definitions.items()
#         ) + "\n\n"

#     examples_text = ""
#     for label in labels:
#         if label not in df.columns:
#             examples_text += f"\n⚠️ Skipping label '{label}' — column not found in dataframe.\n"
#             continue

#         trigger_examples = df[df[label].fillna(0) == 1]
#         if current_index is not None:
#             trigger_examples = trigger_examples[trigger_examples.index != current_index]

#         selected_indices, selected_scores = [], []

#         # Case 1: Use precomputed similar indices from JSON
#         if current_index is not None and str(current_index) in similar_post_map:
#             similar_ids_with_scores = [(int(idx), float(score)) for idx, score in similar_post_map[str(current_index)]]
#             for idx, score in similar_ids_with_scores:
#                 if idx in trigger_examples.index:
#                     selected_indices.append(idx)
#                     selected_scores.append(score)
#                 if len(selected_indices) == k:
#                     break
#         else:
#             # Case 2: Compute cosine similarity from scratch
#             if not trigger_examples.empty:
#                 example_indices = trigger_examples.index.tolist()
#                 example_embeddings = embeddings[example_indices]
#                 sims = cosine_similarity([input_embedding], example_embeddings)[0]
#                 top_k_idx = np.argsort(sims)[-k:][::-1]
#                 selected_indices = [example_indices[i] for i in top_k_idx]
#                 selected_scores = [sims[i] for i in top_k_idx]

#         examples_text += f"\n🔖 Label: {label}\n"
#         examples_text += f"Definition: {definitions[label].strip()}\n"

#         # 🧠 Store collected examples for this label
#         collected_examples_per_label[label] = []

#         if selected_indices:
#             examples_text += f"Here are {len(selected_indices)} similar examples labeled as '{label}':\n"
#             for idx, score in zip(selected_indices, selected_scores):
#                 row = df.loc[idx]
#                 example_title = row.get("title", "[No Title]")
#                 example_body = row.get("body", "[No Body]")

#                 # Store for external access later
#                 collected_examples_per_label[label].append({
#                     "index": idx,
#                     "score": round(score, 4),
#                     "title": example_title,
#                     "body": example_body
#                 })

#                 if debug:
#                     print(f"\n🔎 Example used for label: {label}")
#                     print(f"📌 Index: {idx}")
#                     print(f"🧠 Similarity Score: {score:.4f}")
#                     print(f"📝 Title: {example_title}")
#                     print(f"📄 Body:\n{example_body}")
#                     print("-" * 100)

#                 examples_text += (
#                     f"- Example Index: {idx} (Score: {score:.4f})\n"
#                     f"  • Title: {example_title}\n"
#                     f"  • Description: {example_body}\n"
#                 )
#         else:
#             examples_text += f"No high similarity examples available for label: {label}\n"

#         examples_text += (
#             "\nStep-by-step Evaluation:\n"
#             f"1. Read the post below carefully.\n"
#             f"2. Compare it with the above definition and examples.\n"
#             f"3. Think about whether the post clearly matches the '{label}' definition.\n"
#             f"4. If YES, write: YES\n"
#             f"5. If NO or unsure, write: NO\n"
#             "Answer:"
#         )
#         examples_text += "\n\n" + "-" * 50 + "\n\n"

#     post_section = (
#         "📌 NEW POST TO EVALUATE:\n"
#         f"Title: {title.strip()}\n"
#         f"Description: {body.strip()}\n\n"
#     )

#     final_instruction = (
#         "Now go through each label above and provide a YES or NO answer as per the instructions.\n"
#         "You may say YES to multiple labels or just one.\n"
#         "If none of the labels apply, then all answers should be NO and the label 'Not Applicable' should be marked as YES.\n\n"
#         "Final Format:\n"
#         "Labels: [comma-separated list of applicable labels OR Not Applicable]"
#     )

#     full_prompt = instr + defs_text + examples_text + post_section + final_instruction
#     return full_prompt

from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import torch
import json
import pandas as pd

# ---- Globals / setup (same assumptions as your code) ----
global collected_examples_per_label
collected_examples_per_label = {}  # Reset at each call

# Assumes `labels` (list of label names) and `definitions` (dict[label] -> text) are already defined
df = pd.read_csv("annotations_dataset_with_index.csv")
embeddings = torch.load("post_embeddings.pt")
model = SentenceTransformer("all-mpnet-base-v2")
with open("similar_posts_k10.json", "r") as f:
    similar_post_map = json.load(f)


def make_multishot_prompt_cosine(k, title, body, current_index=None, include_defs=True, debug=False):
    """
    Builds one prompt per label using the EXACT template the user provided.
    Also computes top-k similar positive examples per label to populate
    `collected_examples_per_label` for external inspection, but DOES NOT
    insert examples into the prompt (to keep it exact).
    Returns: dict[label] -> prompt_str
    """
    global collected_examples_per_label
    collected_examples_per_label = {}  # reset on each call

    input_text = (title or "") + " " + (body or "")
    input_embedding = model.encode([input_text])[0]

    prompts_by_label = {}

    for label_name in labels:
        # ---- Gather positive examples for this label (for external use only) ----
        if label_name not in df.columns:
            # Keep structure predictable even if column missing
            collected_examples_per_label[label_name] = []
            if debug:
                print(f"⚠️ Skipping label '{label_name}' — column not found in dataframe.")
        else:
            trigger_examples = df[df[label_name].fillna(0) == 1]
            if current_index is not None:
                trigger_examples = trigger_examples[trigger_examples.index != current_index]

            selected_indices, selected_scores = [], []

            # Case 1: Use precomputed similar indices from JSON (if available for this post)
            used_json = False
            if current_index is not None and str(current_index) in similar_post_map:
                used_json = True
                similar_ids_with_scores = [(int(idx), float(score)) for idx, score in similar_post_map[str(current_index)]]
                for idx, score in similar_ids_with_scores:
                    if idx in trigger_examples.index:
                        selected_indices.append(idx)
                        selected_scores.append(score)
                    if len(selected_indices) == k:
                        break

            # Case 2: Compute similarity if needed
            if not selected_indices and not trigger_examples.empty:
                example_indices = trigger_examples.index.tolist()
                example_embeddings = embeddings[example_indices]
                sims = cosine_similarity([input_embedding], example_embeddings)[0]
                top_k_idx = np.argsort(sims)[-k:][::-1]
                selected_indices = [example_indices[i] for i in top_k_idx]
                selected_scores = [float(sims[i]) for i in top_k_idx]

            # Store for external consumption
            collected_examples_per_label[label_name] = []
            for idx, score in zip(selected_indices, selected_scores):
                row = df.loc[idx]
                collected_examples_per_label[label_name].append({
                    "index": int(idx),
                    "score": round(score, 4),
                    "title": row.get("title", "[No Title]"),
                    "body": row.get("body", "[No Body]")
                })
                if debug:
                    print(f"\n🔎 Example for '{label_name}' "
                          f"(source={'json' if used_json else 'cosine'}): idx={idx} score={score:.4f}")
                    print(f"• Title: {row.get('title', '[No Title]')}")
                    print(f"• Body: {row.get('body', '[No Body]')[:300]}...")
                    print("-" * 80)

        # ---- Build the EXACT prompt string for this label ----
        definition = definitions.get(label_name, "").strip()
        # Note: do not add or alter wording; keep exactly as provided
        prompt_str = (
            "You are an attentive content reviewer tasked with identifying specific emotional or traumatic triggers "
            "in Reddit posts related to abortion, miscarriage, or sexual violence.\n\n"
            "Your task is to analyze the post and decide whether readers could be triggered by it. "
            "We have multiple categories of trigger warnings, but in this case, "
            "you will evaluate **only one trigger label at a time** using a careful step-by-step approach.\n\n"
            f"Trigger Warning Label to evaluate: *{label_name}*\n\n"
            f"Definition of '{label_name}':\n"
            f"{definition}\n\n"
            "Step-by-step instructions:\n"
            "1. Carefully read the post (Title and Description).\n"
            "2. Decide whether the post *explicitly and clearly matches* the above definition.\n"
            "3. If the post is vague, uncertain, or not directly related to the label, answer NO.\n"
            "4. If there is strong, explicit evidence that matches the definition, answer YES.\n"
            "5. Do not infer, assume, or guess based on weak signals.\n\n"
            f"Title: {str(title).strip()}\n\n"
            f"Description: {str(body).strip()}\n\n"
            "Answer with only YES or NO (uppercase, no punctuation, no explanation).\n"
            f"Does this post contain the '{label_name}' trigger?\n"
            "Answer:"
        )

        prompts_by_label[label_name] = prompt_str

    return prompts_by_label


# (Optional) Convenience helper if you want a single label’s prompt directly
def make_prompt_for_label(label_name, k, title, body, current_index=None, debug=False):
    pm = make_multishot_prompt_cosine(k, title, body, current_index=current_index, debug=debug)
    return pm[label_name]



In [25]:
# Cell 8 : Parse Trigger Labels

def parse_trigger_labels(llm_response):
    """
    Convert LLM YES/NO responses to 0/1 values for each trigger label.
    Assumes answers are given in the format:
    Is it a possible trigger for [Label]? YES/NO
    """
    result = {}
    for label in labels:
        # Create a matching string like "trigger for Abuse"
        keyword = f"trigger for {label.lower()}"
        # Search line-by-line
        for line in llm_response.splitlines():
            if keyword in line.lower():
                if "yes" in line.lower():
                    result[label] = 1
                else:
                    result[label] = 0
                break
        else:
            # If label line isn't found at all, mark it as 0
            result[label] = 0
    return result

## GPT Function and output

### OPENAI-API

In [ ]:
# Cell 9 : OpenAI Version Check and Compatibility Fix
import openai
import os
from dotenv import load_dotenv  # ✅ Correct syntax
from packaging import version

# Load environment variables from .env
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("🔍 Checking OpenAI configuration...")
print(f"OpenAI version: {openai.__version__}")

# Determine API format using version
if version.parse(openai.__version__) >= version.parse("1.0.0"):
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    api_format = "modern"
    print("✅ Using modern OpenAI API format (v1.x)")
else:
    openai.api_key = OPENAI_API_KEY
    api_format = "legacy"
    print("✅ Using legacy OpenAI API format (v0.x)")

print(f"🔧 API Format: {api_format}")
print("🎯 OpenAI setup complete!")


🔍 Checking OpenAI configuration...
OpenAI version: 1.99.1
✅ Using modern OpenAI API format (v1.x)
🔧 API Format: modern
🎯 OpenAI setup complete!


In [ ]:
# Cell 10 : Get Trigger Warnings from LLM

import time
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load API key from .env
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Initialize modern OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

def getTriggerWarningsLLMResponse(post_text, model="gpt-4o", temperature=0, max_tokens=512, verbose=True):
    """
    Classify trigger warnings for a given post using OpenAI (modern v1.x API).

    Args:
        post_text (str): Text (title + body) of the post to classify.
        model (str): OpenAI model name (default: gpt-4-1106-preview)
        temperature (float): Sampling randomness (default: 0)
        max_tokens (int): Max tokens to allow in the response
        verbose (bool): Whether to print detailed logs

    Returns:
        str: The LLM's raw response text (classification result), or None on failure.
    """
    if verbose:
        print("🧠 Classifying post for trigger warnings...")
        print("=" * 60)

    max_retries = 3
    base_delay = 60  # seconds

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "user",
                        "content": post_text
                    }
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )

            llm_response = response.choices[0].message.content

            if verbose:
                print("✅ LLM Response received:")
                print("-" * 60)

            return llm_response

        except Exception as e:
            error_msg = str(e).lower()

            if "rate limit" in error_msg or "429" in error_msg:
                wait_time = base_delay * (2 ** attempt)
                if verbose:
                    print(f"⏳ Rate limit hit. Retrying in {wait_time}s... (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
                continue
            elif "quota" in error_msg or "billing" in error_msg:
                if verbose:
                    print(f"💳 Quota/billing issue: {e}")
                return None
            else:
                if verbose:
                    print(f"❌ Error: {e}")
                return None

    if verbose:
        print("❌ Failed after multiple retries due to persistent issues.")
    return None


# ✅ Confirmation message
print("✅ getTriggerWarningsLLMResponse() ready for multi-label classification!")
print("📋 Use: response = getTriggerWarningsLLMResponse(post_text)")


✅ getTriggerWarningsLLMResponse() ready for multi-label classification!
📋 Use: response = getTriggerWarningsLLMResponse(post_text)


### Zero-Shot

In [ ]:
# Cell 11 : Zero shot random post no.1
import time
import pandas as pd

# Sample post for evaluation
title = "Relationship fight and sudden bleeding"
sample_post = "My boyfriend and I had a fight and I started bleeding heavily that night."
body = sample_post

# Store results
final_predictions = {}

# Loop through each label and run LLM on its prompt
for label in labels:
    print(f"\n🧠 Evaluating Label: {label}")
    definition = definitions[label]
    single_prompt = make_per_label_prompt(title, body, label, definition)
    response = getTriggerWarningsLLMResponse(single_prompt)
    
    answer = response.strip().upper() if response else "NO"
    final_predictions[label] = answer

# 🩵 Force fallback to "Not Applicable" if nothing was flagged
if all(value == "NO" for key, value in final_predictions.items() if key != "Not Applicable"):
    final_predictions["Not Applicable"] = "YES"

# 🔖 Final Multi-label Predictions:
final_labels = []

for label in labels:
    response = final_predictions.get(label, "").strip().upper()
    if response == "YES":
        final_labels.append(label)

if not final_labels:
    final_labels = ["Not Applicable"]

print(f"\n🔖 Final Multi-label Prediction:\nLabels: {final_labels}")



🧠 Evaluating Label: Medical
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------

🧠 Evaluating Label: Mental Health
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------

🧠 Evaluating Label: Abuse
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------

🧠 Evaluating Label: Aggression
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------

🧠 Evaluating Label: Sexual
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------

🧠 Evaluating Label: Discrimination
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------

🧠 Evaluating Label: Pregnancy
🧠 Classi

In [ ]:
# Cell 12 : Zero shot all 75 posts prompt 1
# Cell: Batch 0-shot labeling for 75 posts (title+body) -> CSV
import pandas as pd
import time, re, json

# Assumes these already exist in your environment:
# - labels: List[str] (includes "Not Applicable" or not — both fine)
# - definitions: Dict[str, str]
# - make_per_label_prompt(title, body, label, definition)
# - getTriggerWarningsLLMResponse(prompt)

INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"
OUTPUT_CSV = "predicted_labels_75_prompt_1.csv"

def normalize_yes_no(text: str) -> str:
    """Parse any LLM response and return 'YES' or 'NO' (default NO)."""
    if not text:
        return "NO"
    s = str(text).strip()
    m = re.search(r"\b(YES|NO)\b", s, flags=re.I)
    if m:
        return m.group(1).upper()
    # fallback: first char heuristic
    return "YES" if s.upper().startswith("Y") else "NO"

def predict_for_post(title: str, body: str):
    """Run per-label 0-shot prompts and return final labels + per-label decisions."""
    per_label = {}
    for label in labels:
        definition = definitions[label]
        prompt = make_per_label_prompt(title, body, label, definition)
        resp = getTriggerWarningsLLMResponse(prompt)
        per_label[label] = normalize_yes_no(resp)
        # tiny pause to be gentle with rate limits (tune/remove as needed)
        time.sleep(0.05)

    # Build final labels: if none are YES (excluding NA), mark Not Applicable
    positives = [lbl for lbl, ans in per_label.items() if ans == "YES" and lbl != "Not Applicable"]
    final_labels = positives if positives else ["Not Applicable"]
    return final_labels, per_label

# --- Load dataset (expects 'title' and 'body' columns) ---
df = pd.read_csv(INPUT_CSV)
df = df[['title', 'body']].fillna('').astype(str)

subset = df.head(75).copy()

predicted_lists = []
per_label_json  = []

for i, (_, row) in enumerate(subset.iterrows(), start=1):
    final_labels, per_label = predict_for_post(row['title'], row['body'])
    predicted_lists.append(final_labels)
    per_label_json.append(json.dumps(per_label, ensure_ascii=False))
    print(f"{i}. Labels: {final_labels}")

# Save results to CSV (labels as JSON strings for easy downstream parsing)
out = subset.copy()
out['PredictedLabels'] = [json.dumps(x, ensure_ascii=False) for x in predicted_lists]
out['PerLabelYESNO']   = per_label_json
out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"\nSaved predictions to {OUTPUT_CSV}")


🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
-----------------------------

In [ ]:
# Cell 13 : Zero shot all 75 posts prompt 2
# Cell: Batch 0-shot labeling for 75 posts (title+body) -> CSV
import pandas as pd
import time, re, json

# Assumes these already exist in your environment:
# - labels: List[str] (includes "Not Applicable" or not — both fine)
# - definitions: Dict[str, str]
# - make_per_label_prompt(title, body, label, definition)
# - getTriggerWarningsLLMResponse(prompt)

INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"
OUTPUT_CSV = "predicted_labels_75_prompt_2.csv"

def normalize_yes_no(text: str) -> str:
    """Parse any LLM response and return 'YES' or 'NO' (default NO)."""
    if not text:
        return "NO"
    s = str(text).strip()
    m = re.search(r"\b(YES|NO)\b", s, flags=re.I)
    if m:
        return m.group(1).upper()
    # fallback: first char heuristic
    return "YES" if s.upper().startswith("Y") else "NO"

def predict_for_post(title: str, body: str):
    """Run per-label 0-shot prompts and return final labels + per-label decisions."""
    per_label = {}
    for label in labels:
        definition = definitions[label]
        prompt = make_per_label_prompt(title, body, label, definition)
        resp = getTriggerWarningsLLMResponse(prompt)
        per_label[label] = normalize_yes_no(resp)
        # tiny pause to be gentle with rate limits (tune/remove as needed)
        time.sleep(0.05)

    # Build final labels: if none are YES (excluding NA), mark Not Applicable
    positives = [lbl for lbl, ans in per_label.items() if ans == "YES" and lbl != "Not Applicable"]
    final_labels = positives if positives else ["Not Applicable"]
    return final_labels, per_label

# --- Load dataset (expects 'title' and 'body' columns) ---
df = pd.read_csv(INPUT_CSV)
df = df[['title', 'body']].fillna('').astype(str)

subset = df.head(75).copy()

predicted_lists = []
per_label_json  = []

for i, (_, row) in enumerate(subset.iterrows(), start=1):
    final_labels, per_label = predict_for_post(row['title'], row['body'])
    predicted_lists.append(final_labels)
    per_label_json.append(json.dumps(per_label, ensure_ascii=False))
    print(f"{i}. Labels: {final_labels}")

# Save results to CSV (labels as JSON strings for easy downstream parsing)
out = subset.copy()
out['PredictedLabels'] = [json.dumps(x, ensure_ascii=False) for x in predicted_lists]
out['PerLabelYESNO']   = per_label_json
out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"\nSaved predictions to {OUTPUT_CSV}")


🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
------------------------------------------------------------
🧠 Classifying post for trigger warnings...
✅ LLM Response received:
-----------------------------

In [ ]:
# Cell 14 : Precision/Recall/F1 per post for prompt 1
# Cell: Precision/Recall/F1 per post using GT "Tags" column (Ubuntu-friendly)
import os
import ast
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# ---- paths (edit if your files live elsewhere) ----
INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"   # ground truth with a 'Tags' column
OUTPUT_CSV = "predicted_labels_75_prompt_1.csv"                      # predictions with 'PredictedLabels' column
SAVE_AS    = "per_post_metrics_from_tags_Prompt_1.csv"                         # will be saved in the current folder

# ---- load data ----
gt_df   = pd.read_csv(INPUT_CSV)
pred_df = pd.read_csv(OUTPUT_CSV)

# align to first 75 rows (adjust if you want more)
gt_subset   = gt_df.head(75).reset_index(drop=True)
pred_subset = pred_df.head(75).reset_index(drop=True)

# ---- helpers ----
def parse_listish(value):
    """
    Parse labels that might be stored as python-list strings or comma-separated strings.
    Returns a set of normalized labels.
    """
    if pd.isna(value):
        return set()
    s = str(value).strip()
    # try python list literal first
    try:
        maybe_list = ast.literal_eval(s)
        if isinstance(maybe_list, (list, tuple)):
            items = maybe_list
        else:
            items = [s]
    except Exception:
        # fallback: comma-separated
        items = [tok.strip() for tok in s.split(",") if tok.strip()]
    # normalize labels (consistent spacing/case)
    return set([str(x).strip() for x in items if str(x).strip()])

# ---- parse labels ----
if "Tags" not in gt_subset.columns:
    raise KeyError("Ground truth CSV must contain a 'Tags' column with human-annotated labels.")

gt_labels_list   = gt_subset["Tags"].apply(parse_listish).tolist()
pred_labels_list = pred_subset["PredictedLabels"].apply(parse_listish).tolist()

# build label universe
all_labels = sorted(list(set().union(*gt_labels_list, *pred_labels_list)))

# ---- compute metrics ----
rows = []
y_true_all, y_pred_all = [], []

for idx, (gt_set, pred_set) in enumerate(zip(gt_labels_list, pred_labels_list), start=1):
    y_true = [1 if l in gt_set else 0 for l in all_labels]
    y_pred = [1 if l in pred_set else 0 for l in all_labels]

    y_true_all.extend(y_true)
    y_pred_all.extend(y_pred)

    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)

    rows.append({
        "PostIndex": idx,
        "TrueLabels": sorted(list(gt_set)),
        "PredictedLabels": sorted(list(pred_set)),
        "Precision": round(p, 3),
        "Recall": round(r, 3),
        "F1": round(f, 3),
    })

metrics_df = pd.DataFrame(rows)

# macro averages = mean of per-post metrics
macro_p = float(metrics_df["Precision"].mean())
macro_r = float(metrics_df["Recall"].mean())
macro_f = float(metrics_df["F1"].mean())

# micro averages = computed on pooled one-hot vectors
micro_p = precision_score(y_true_all, y_pred_all, zero_division=0)
micro_r = recall_score(y_true_all, y_pred_all, zero_division=0)
micro_f = f1_score(y_true_all, y_pred_all, zero_division=0)

summary_rows = pd.DataFrame([
    {"PostIndex": "Macro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(macro_p, 3), "Recall": round(macro_r, 3), "F1": round(macro_f, 3)},
    {"PostIndex": "Micro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(micro_p, 3), "Recall": round(micro_r, 3), "F1": round(micro_f, 3)},
])

final_metrics = pd.concat([metrics_df, summary_rows], ignore_index=True)

# ---- save (no /mnt/data, so always local) ----
final_metrics.to_csv(SAVE_AS, index=False, encoding="utf-8")
print(f"✅ Saved per-post metrics + macro/micro averages to: {os.path.abspath(SAVE_AS)}")


✅ Saved per-post metrics + macro/micro averages to: /home/ubuntu/per_post_metrics_from_tags_Prompt_1.csv


In [ ]:
# Cell 15 : Precision/Recall/F1 per post for prompt 2
# Cell: Precision/Recall/F1 per post using GT "Tags" column (Ubuntu-friendly)
import os
import ast
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# ---- paths (edit if your files live elsewhere) ----
INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"   # ground truth with a 'Tags' column
OUTPUT_CSV = "predicted_labels_75_prompt_2.csv"                      # predictions with 'PredictedLabels' column
SAVE_AS    = "per_post_metrics_from_tags_Prompt_2.csv"                         # will be saved in the current folder

# ---- load data ----
gt_df   = pd.read_csv(INPUT_CSV)
pred_df = pd.read_csv(OUTPUT_CSV)

# align to first 75 rows (adjust if you want more)
gt_subset   = gt_df.head(75).reset_index(drop=True)
pred_subset = pred_df.head(75).reset_index(drop=True)

# ---- helpers ----
def parse_listish(value):
    """
    Parse labels that might be stored as python-list strings or comma-separated strings.
    Returns a set of normalized labels.
    """
    if pd.isna(value):
        return set()
    s = str(value).strip()
    # try python list literal first
    try:
        maybe_list = ast.literal_eval(s)
        if isinstance(maybe_list, (list, tuple)):
            items = maybe_list
        else:
            items = [s]
    except Exception:
        # fallback: comma-separated
        items = [tok.strip() for tok in s.split(",") if tok.strip()]
    # normalize labels (consistent spacing/case)
    return set([str(x).strip() for x in items if str(x).strip()])

# ---- parse labels ----
if "Tags" not in gt_subset.columns:
    raise KeyError("Ground truth CSV must contain a 'Tags' column with human-annotated labels.")

gt_labels_list   = gt_subset["Tags"].apply(parse_listish).tolist()
pred_labels_list = pred_subset["PredictedLabels"].apply(parse_listish).tolist()

# build label universe
all_labels = sorted(list(set().union(*gt_labels_list, *pred_labels_list)))

# ---- compute metrics ----
rows = []
y_true_all, y_pred_all = [], []

for idx, (gt_set, pred_set) in enumerate(zip(gt_labels_list, pred_labels_list), start=1):
    y_true = [1 if l in gt_set else 0 for l in all_labels]
    y_pred = [1 if l in pred_set else 0 for l in all_labels]

    y_true_all.extend(y_true)
    y_pred_all.extend(y_pred)

    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)

    rows.append({
        "PostIndex": idx,
        "TrueLabels": sorted(list(gt_set)),
        "PredictedLabels": sorted(list(pred_set)),
        "Precision": round(p, 3),
        "Recall": round(r, 3),
        "F1": round(f, 3),
    })

metrics_df = pd.DataFrame(rows)

# macro averages = mean of per-post metrics
macro_p = float(metrics_df["Precision"].mean())
macro_r = float(metrics_df["Recall"].mean())
macro_f = float(metrics_df["F1"].mean())

# micro averages = computed on pooled one-hot vectors
micro_p = precision_score(y_true_all, y_pred_all, zero_division=0)
micro_r = recall_score(y_true_all, y_pred_all, zero_division=0)
micro_f = f1_score(y_true_all, y_pred_all, zero_division=0)

summary_rows = pd.DataFrame([
    {"PostIndex": "Macro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(macro_p, 3), "Recall": round(macro_r, 3), "F1": round(macro_f, 3)},
    {"PostIndex": "Micro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(micro_p, 3), "Recall": round(micro_r, 3), "F1": round(micro_f, 3)},
])

final_metrics = pd.concat([metrics_df, summary_rows], ignore_index=True)

# ---- save (no /mnt/data, so always local) ----
final_metrics.to_csv(SAVE_AS, index=False, encoding="utf-8")
print(f"✅ Saved per-post metrics + macro/micro averages to: {os.path.abspath(SAVE_AS)}")


✅ Saved per-post metrics + macro/micro averages to: /home/ubuntu/per_post_metrics_from_tags_Prompt_2.csv


In [65]:
# 🛠️ Step 1: Clean 'Tags' column if needed
df["Tags"] = df["Tags"].apply(
    lambda x: [tag.strip() for tag in x.split(",")] if isinstance(x, str) else x
)

# 🧷 Step 2: Create binary columns per label
all_labels = [
    "Medical",
    "Mental Health",
    "Abuse",
    "Aggression",
    "Sexual",
    "Discrimination",
    "Pregnancy",
    "Not Applicable"
]

for label in all_labels:
    df[label] = df["Tags"].apply(lambda tags: int(label in tags if isinstance(tags, list) else []))


In [66]:
print(df.columns.tolist())

['id', 'soure', 'subreddit', 'title', 'body', 'created_utc', 'url', 'Tags', 'text', 'Medical', 'Mental Health', 'Abuse', 'Aggression', 'Sexual', 'Discrimination', 'Pregnancy', 'Not Applicable']


### Multi-Shot

In [70]:
# Initialize global variable for examples
global collected_examples_per_label

# Input post
title = "Bleeding After an Argument — Scared and Confused"
body = (
    "I'm really scared and don't know who else to talk to. My boyfriend and I had a horrible argument last night — he yelled at me for over an hour, and I ended up locking myself in the bathroom crying. "
    "Later that night, I noticed I was bleeding heavily, way more than my normal period, and I felt this deep cramping that wouldn't go away. "
    "I'm around 6 weeks late on my period and had been suspecting I might be pregnant, but I hadn’t taken a test yet because I was afraid of what it might say. "
    "Now, I don't know if this is a miscarriage or something else. I feel so alone. I’ve been reading online and some posts say it could just be stress-induced bleeding or maybe even implantation bleeding, but this doesn't feel like that. "
    "I don’t know whether to go to the hospital. I’m scared he might get angry again if he finds out I’m talking to anyone about this. "
    "Has anyone experienced something similar? Could stress cause this much bleeding? Or could this have been a miscarriage? I'm really not sure how to cope with everything happening right now."
)

# Generate prompt with cosine-based similar examples
prompt = make_multishot_prompt_cosine(
    k=3,
    title=title,
    body=body,
    current_index=None,
    include_defs=True,
    debug=False
)

# Get LLM response
response = getTriggerWarningsLLMResponse(prompt, verbose=False)

import re, ast, json, numpy as np

def extract_labels_from_qa(text, allowed_labels=None):
    pattern = re.compile(r"Label:\s*([^\n]+).*?Answer:\s*(YES|NO)", re.S | re.I)
    pairs = pattern.findall(text)
    picked = []
    allowed = set(allowed_labels) if allowed_labels else None
    for raw_label, ans in pairs:
        label = raw_label.replace("🔖", "").strip()
        if allowed is not None and label not in allowed:
            continue
        if ans.strip().upper() == "YES":
            picked.append(label)
    return picked

def parse_labels_bracketed(response_text: str):
    if not response_text:
        return []
    m = re.search(r"[Ll]abels\s*[:\-]?\s*\[([^\]]*)\]", response_text, flags=re.S)
    if m:
        inner = m.group(1).strip()
        try:
            candidate = "[" + inner + "]"
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed]
        except Exception:
            return [tok.strip(" '\"\n\t") for tok in inner.split(",") if tok.strip()]
    m2 = re.search(r"\[([^\]]+)\]", response_text, flags=re.S)
    if m2:
        try:
            candidate = "[" + m2.group(1) + "]"
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed]
        except Exception:
            pass
    parts = re.split(r"[Ll]abels\s*[:\-]?", response_text)
    if len(parts) > 1:
        tail = parts[-1]
        tail_line = tail.splitlines()[0] if "\n" in tail else tail
        if "[" in tail_line and "]" in tail_line:
            inner = tail_line[tail_line.find("[")+1 : tail_line.find("]")]
            return [tok.strip(" '\"\n\t") for tok in inner.split(",") if tok.strip()]
        else:
            tokens = [t.strip(" '\"\n\t") for t in tail_line.split(",") if t.strip()]
            if tokens:
                return tokens
    return []

try:
    allowed_label_list = labels
except NameError:
    allowed_label_list = None

predicted_labels = extract_labels_from_qa(response, allowed_labels=allowed_label_list)

if not predicted_labels:
    predicted_labels = parse_labels_bracketed(response)

if not predicted_labels:
    predicted_labels = ["Not Applicable"]

# ✅ Only print labels
print(predicted_labels)

# Save similar examples to JSON
def convert_to_serializable(obj):
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return str(obj)

with open("used_examples_for_latest_post.json", "w") as f:
    json.dump(collected_examples_per_label, f, indent=2, default=convert_to_serializable)



['Medical', 'Mental Health', 'Abuse', 'Pregnancy']


In [69]:
# Initialize global variable for examples
global collected_examples_per_label

# Input post
title = "Relationship fight and sudden bleeding"
body = "My boyfriend and I had a fight and I started bleeding heavily that night."

# Generate prompt with cosine-based similar examples
prompt = make_multishot_prompt_cosine(
    k=3,
    title=title,
    body=body,
    current_index=None,
    include_defs=True,
    debug=False
)

# Get LLM response
response = getTriggerWarningsLLMResponse(prompt, verbose=False)

import re, ast, json, numpy as np

def extract_labels_from_qa(text, allowed_labels=None):
    pattern = re.compile(r"Label:\s*([^\n]+).*?Answer:\s*(YES|NO)", re.S | re.I)
    pairs = pattern.findall(text)
    picked = []
    allowed = set(allowed_labels) if allowed_labels else None
    for raw_label, ans in pairs:
        label = raw_label.replace("🔖", "").strip()
        if allowed is not None and label not in allowed:
            continue
        if ans.strip().upper() == "YES":
            picked.append(label)
    return picked

def parse_labels_bracketed(response_text: str):
    if not response_text:
        return []
    m = re.search(r"[Ll]abels\s*[:\-]?\s*\[([^\]]*)\]", response_text, flags=re.S)
    if m:
        inner = m.group(1).strip()
        try:
            candidate = "[" + inner + "]"
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed]
        except Exception:
            return [tok.strip(" '\"\n\t") for tok in inner.split(",") if tok.strip()]
    m2 = re.search(r"\[([^\]]+)\]", response_text, flags=re.S)
    if m2:
        try:
            candidate = "[" + m2.group(1) + "]"
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed]
        except Exception:
            pass
    parts = re.split(r"[Ll]abels\s*[:\-]?", response_text)
    if len(parts) > 1:
        tail = parts[-1]
        tail_line = tail.splitlines()[0] if "\n" in tail else tail
        if "[" in tail_line and "]" in tail_line:
            inner = tail_line[tail_line.find("[")+1 : tail_line.find("]")]
            return [tok.strip(" '\"\n\t") for tok in inner.split(",") if tok.strip()]
        else:
            tokens = [t.strip(" '\"\n\t") for t in tail_line.split(",") if t.strip()]
            if tokens:
                return tokens
    return []

try:
    allowed_label_list = labels
except NameError:
    allowed_label_list = None

predicted_labels = extract_labels_from_qa(response, allowed_labels=allowed_label_list)

if not predicted_labels:
    predicted_labels = parse_labels_bracketed(response)

if not predicted_labels:
    predicted_labels = ["Not Applicable"]

# ✅ Only print labels
print(predicted_labels)

# Save similar examples to JSON
def convert_to_serializable(obj):
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return str(obj)

with open("used_examples_for_latest_post.json", "w") as f:
    json.dump(collected_examples_per_label, f, indent=2, default=convert_to_serializable)


['Not Applicable']


In [73]:
# Initialize global variable for examples
global collected_examples_per_label

# Input post
# title = "Complications after abortion?"
# body = "Hi everyone, Ive read that abortions don’t cause infertility. There is a part of me that doesn’t fully believe that. Could be my anxiety speaking. Since my abortion (August 2024), I’ve been bleeding half way through my cycle and after sex, brown spotting, and recently my periods have been late (last three months) and my uterus, ovaries and cervix are so tender it hurts to walk, bend over or even apply a bit of pressure. It isn’t unbearable pain, it’s manageable but rn it’s at its worst and it’s still out of the normal for me. Anyone know what it possibly could be? I have a gynecologist appointment in about 3 weeks but I’m hurting now and I’m getting very worried."

# Generate prompt with cosine-based similar examples
prompt = make_multishot_prompt_cosine(
    k=3,
    title=df.loc[0, "title"],
    body=df.loc[0, "body"],
    current_index=0,      # <-- crucial
    include_defs=True,
    debug=False
)

# Get LLM response
response = getTriggerWarningsLLMResponse(prompt, verbose=False)

import re, ast, json, numpy as np

def extract_labels_from_qa(text, allowed_labels=None):
    pattern = re.compile(r"Label:\s*([^\n]+).*?Answer:\s*(YES|NO)", re.S | re.I)
    pairs = pattern.findall(text)
    picked = []
    allowed = set(allowed_labels) if allowed_labels else None
    for raw_label, ans in pairs:
        label = raw_label.replace("🔖", "").strip()
        if allowed is not None and label not in allowed:
            continue
        if ans.strip().upper() == "YES":
            picked.append(label)
    return picked

def parse_labels_bracketed(response_text: str):
    if not response_text:
        return []
    m = re.search(r"[Ll]abels\s*[:\-]?\s*\[([^\]]*)\]", response_text, flags=re.S)
    if m:
        inner = m.group(1).strip()
        try:
            candidate = "[" + inner + "]"
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed]
        except Exception:
            return [tok.strip(" '\"\n\t") for tok in inner.split(",") if tok.strip()]
    m2 = re.search(r"\[([^\]]+)\]", response_text, flags=re.S)
    if m2:
        try:
            candidate = "[" + m2.group(1) + "]"
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return [str(x).strip() for x in parsed]
        except Exception:
            pass
    parts = re.split(r"[Ll]abels\s*[:\-]?", response_text)
    if len(parts) > 1:
        tail = parts[-1]
        tail_line = tail.splitlines()[0] if "\n" in tail else tail
        if "[" in tail_line and "]" in tail_line:
            inner = tail_line[tail_line.find("[")+1 : tail_line.find("]")]
            return [tok.strip(" '\"\n\t") for tok in inner.split(",") if tok.strip()]
        else:
            tokens = [t.strip(" '\"\n\t") for t in tail_line.split(",") if t.strip()]
            if tokens:
                return tokens
    return []

try:
    allowed_label_list = labels
except NameError:
    allowed_label_list = None

predicted_labels = extract_labels_from_qa(response, allowed_labels=allowed_label_list)

if not predicted_labels:
    predicted_labels = parse_labels_bracketed(response)

if not predicted_labels:
    predicted_labels = ["Not Applicable"]

# ✅ Only print labels
print(predicted_labels)

# Save similar examples to JSON
def convert_to_serializable(obj):
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return str(obj)

with open("used_examples_for_latest_post.json", "w") as f:
    json.dump(collected_examples_per_label, f, indent=2, default=convert_to_serializable)


['Medical', 'Pregnancy']


In [73]:
print(df.columns.tolist())


['id', 'soure', 'subreddit', 'title', 'body', 'created_utc', 'url', 'Tags', 'text']


## LLama Function and Output

### LLama Model

In [6]:
from together import Together
import os
from dotenv import load_dotenv

load_dotenv()
client = Together(api_key=os.environ["TOGETHER_API_KEY"])

resp = client.chat.completions.create(
    model="meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo",  # <-- corrected
    messages=[{"role": "user", "content": "Hello from EC2!"}],
    max_tokens=50,
)
print(resp.choices[0].message.content)


Hello from the other side. How's life on EC2 treating you? Are you working on a project or just exploring the AWS ecosystem?


In [7]:
import os
import time
from dotenv import load_dotenv
from together import Together

# One-time setup
load_dotenv()
client = Together(api_key=os.environ["TOGETHER_API_KEY"])
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo"  # change if needed

def getTriggerWarningsLLMResponse_Llama(
    post_text,
    *,
    temperature: float = 0.0,
    max_tokens: int = 512,
    top_p: float = 1.0,
    repetition_penalty: float = 1.1,
    verbose: bool = True,
    retries: int = 3,
    base_delay: float = 8.0,
    **kwargs,  # tolerate extras like mode="raw"
):
    """
    Call Together AI (Llama) to classify a single prompt and return raw text.

    Args:
        post_text (str): Fully-formed prompt.
        temperature (float): Sampling temperature.
        max_tokens (int): Max tokens to generate.
        top_p (float): Nucleus sampling.
        repetition_penalty (float): Penalty for repetition.
        verbose (bool): Print simple logs/errors.
        retries (int): Number of retry attempts on errors.
        base_delay (float): Base delay (seconds) for exponential backoff.
        **kwargs: Ignored extra args (e.g., mode="raw") for call-site compatibility.

    Returns:
        Optional[str]: The model's text output, or None if all retries fail.
    """
    # Requires global `client` and `MODEL_ID` to be defined earlier:
    # client = Together(api_key=os.environ["TOGETHER_API_KEY"])
    # MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo"

    if verbose:
        print("🧠 Classifying post (Together AI - Llama)")

    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL_ID,
                messages=[{"role": "user", "content": str(post_text)}],
                temperature=float(temperature),
                max_tokens=int(max_tokens),
                top_p=float(top_p),
                repetition_penalty=float(repetition_penalty),
            )
            # Safely extract the text
            text = None
            if resp and getattr(resp, "choices", None):
                choice0 = resp.choices[0]
                # Together returns .message.content
                if hasattr(choice0, "message") and getattr(choice0.message, "content", None):
                    text = choice0.message.content
                elif getattr(choice0, "text", None):
                    text = choice0.text

            if verbose:
                print("✅ LLM response received.")

            return text if (text is not None and str(text).strip()) else None

        except Exception as e:
            if verbose:
                print(f"❌ Error on attempt {attempt+1}/{retries}: {e}")
            if attempt < retries - 1:
                wait = base_delay * (2 ** attempt)
                if verbose:
                    print(f"⏳ Retrying in {wait:.1f}s...")
                try:
                    import time as _t
                    _t.sleep(wait)
                except Exception:
                    pass
                continue
            else:
                if verbose:
                    print("❌ Giving up after retries.")
                return None

### Example Testing

In [28]:
# Cell 3: Chain-of-thought prompting for each label (LLama backend)
import time
import pandas as pd

# Sample post for evaluation
title = "Relationship fight and sudden bleeding"
sample_post = "My boyfriend and I had a fight and I started bleeding heavily that night."
body = sample_post

# Store results
final_predictions = {}

# Loop through each label and run LLM on its prompt
for label in labels:
    print(f"\n🧠 Evaluating Label: {label}")
    definition = definitions[label]
    single_prompt = make_per_label_prompt(title, body, label, definition)

    # 🔁 Use local Llama instead of OpenAI
    response = getTriggerWarningsLLMResponse_Llama(
        single_prompt,
        temperature=0.0,
        max_tokens=16,    # small, expecting YES/NO
        verbose=False
    )

    # Normalize to strict YES/NO
    answer = (response or "NO").strip().upper()
    answer = "YES" if "YES" in answer else "NO"
    final_predictions[label] = answer

# 🩵 Force fallback to "Not Applicable" if nothing was flagged
if all(value == "NO" for key, value in final_predictions.items() if key != "Not Applicable"):
    final_predictions["Not Applicable"] = "YES"

# 🔖 Final Multi-label Predictions:
final_labels = []
for label in labels:
    response = final_predictions.get(label, "").strip().upper()
    if response == "YES":
        final_labels.append(label)

if not final_labels:
    final_labels = ["Not Applicable"]

print(f"\n🔖 Final Multi-label Prediction:\nLabels: {final_labels}")



🧠 Evaluating Label: Medical

🧠 Evaluating Label: Mental Health

🧠 Evaluating Label: Abuse

🧠 Evaluating Label: Aggression

🧠 Evaluating Label: Sexual

🧠 Evaluating Label: Discrimination

🧠 Evaluating Label: Pregnancy

🧠 Evaluating Label: Not Applicable

🔖 Final Multi-label Prediction:
Labels: ['Not Applicable']


In [34]:
# Cell 3: Chain-of-thought prompting for each label (LLama backend)
import time
import pandas as pd

# Sample post for evaluation
# sample_post = "My boyfriend and I had a fight and I started bleeding heavily that night. It was a miscarriage or my period. But I am very scared about it"
# title = "Relationship fight and sudden bleeding"
# body = sample_post

title = "Bleeding After an Argument — Scared and Confused"
body = (
    "I'm really scared and don't know who else to talk to. My boyfriend and I had a horrible argument last night — he yelled at me for over an hour, and I ended up locking myself in the bathroom crying. "
    "Later that night, I noticed I was bleeding heavily, way more than my normal period, and I felt this deep cramping that wouldn't go away. "
    "I'm around 6 weeks late on my period and had been suspecting I might be pregnant, but I hadn’t taken a test yet because I was afraid of what it might say. "
    "Now, I don't know if this is a miscarriage or something else. I feel so alone. I’ve been reading online and some posts say it could just be stress-induced bleeding or maybe even implantation bleeding, but this doesn't feel like that. "
    "I don’t know whether to go to the hospital. I’m scared he might get angry again if he finds out I’m talking to anyone about this. "
    "Has anyone experienced something similar? Could stress cause this much bleeding? Or could this have been a miscarriage? I'm really not sure how to cope with everything happening right now."
)

# Store results
final_predictions = {}

# Loop through each label and run LLM on its prompt
for label in labels:
    print(f"\n🧠 Evaluating Label: {label}")
    definition = definitions[label]
    single_prompt = make_per_label_prompt(title, body, label, definition)

    # 🔁 Use local Llama instead of OpenAI
    response = getTriggerWarningsLLMResponse_Llama(
        single_prompt,
        temperature=0.0,
        max_tokens=16,    # small, expecting YES/NO
        verbose=False
    )

    # Normalize to strict YES/NO
    answer = (response or "NO").strip().upper()
    answer = "YES" if "YES" in answer else "NO"
    final_predictions[label] = answer

# 🩵 Force fallback to "Not Applicable" if nothing was flagged
if all(value == "NO" for key, value in final_predictions.items() if key != "Not Applicable"):
    final_predictions["Not Applicable"] = "YES"

# 🔖 Final Multi-label Predictions:
final_labels = []
for label in labels:
    response = final_predictions.get(label, "").strip().upper()
    if response == "YES":
        final_labels.append(label)

if not final_labels:
    final_labels = ["Not Applicable"]

print(f"\n🔖 Final Multi-label Prediction:\nLabels: {final_labels}")


🧠 Evaluating Label: Medical

🧠 Evaluating Label: Mental Health

🧠 Evaluating Label: Abuse

🧠 Evaluating Label: Aggression

🧠 Evaluating Label: Sexual

🧠 Evaluating Label: Discrimination

🧠 Evaluating Label: Pregnancy

🧠 Evaluating Label: Not Applicable

🔖 Final Multi-label Prediction:
Labels: ['Medical', 'Mental Health', 'Pregnancy']


### 0 Shot Prompt testing for LLama

In [26]:
# Cell 12 : Zero shot all 75 posts prompt 1
# Cell: Batch 0-shot labeling for 75 posts (title+body) -> CSV
import pandas as pd
import time, re, json

# Assumes these already exist in your environment:
# - labels: List[str] (includes "Not Applicable" or not — both fine)
# - definitions: Dict[str, str]
# - make_per_label_prompt(title, body, label, definition)
# - getTriggerWarningsLLMResponse(prompt)

INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"
OUTPUT_CSV = "predicted_labels_75_0_shot_llama.csv"

def normalize_yes_no(text: str) -> str:
    """Parse any LLM response and return 'YES' or 'NO' (default NO)."""
    if not text:
        return "NO"
    s = str(text).strip()
    m = re.search(r"\b(YES|NO)\b", s, flags=re.I)
    if m:
        return m.group(1).upper()
    # fallback: first char heuristic
    return "YES" if s.upper().startswith("Y") else "NO"

# def predict_for_post(title: str, body: str):
#     """Run per-label 0-shot prompts and return final labels + per-label decisions."""
#     per_label = {}

#     # Evaluate all labels (including NA if it's in your `labels` list)
#     for label in labels:
#         definition = definitions[label]
#         prompt = make_per_label_prompt(title, body, label, definition)
#         resp = getTriggerWarningsLLMResponse_Llama(prompt)
#         per_label[label] = normalize_yes_no(resp)
#         time.sleep(0.05)  # rate-limit friendliness

#     # 1) Collect positives excluding NA
#     positives = [lbl for lbl, ans in per_label.items()
#                  if ans == "YES" and lbl != "Not Applicable"]

#     # 2) Decide final labels with NA as exclusive fallback
#     if positives:
#         # If NA is also YES, force it to be standalone (override)
#         if per_label.get("Not Applicable") == "YES":
#             final_labels = ["Not Applicable"]
#         else:
#             final_labels = positives
#     else:
#         final_labels = ["Not Applicable"]

#     # 3) Extra safety net (handles any future code edits)
#     if "Not Applicable" in final_labels and len(final_labels) > 1:
#         final_labels = ["Not Applicable"]

#     return final_labels, per_label


def predict_for_post(title: str, body: str):
    """Run per-label 0-shot prompts and return final labels + per-label decisions."""
    per_label = {}
    for label in labels:
        definition = definitions[label]
        prompt = make_per_label_prompt(title, body, label, definition)
        resp = getTriggerWarningsLLMResponse_Llama(prompt)
        per_label[label] = normalize_yes_no(resp)
        # tiny pause to be gentle with rate limits (tune/remove as needed)
        time.sleep(0.05)

    # Build final labels: if none are YES (excluding NA), mark Not Applicable
    positives = [lbl for lbl, ans in per_label.items() if ans == "YES" and lbl != "Not Applicable"]
    final_labels = positives if positives else ["Not Applicable"]
    return final_labels, per_label

# --- Load dataset (expects 'title' and 'body' columns) ---
df = pd.read_csv(INPUT_CSV)
df = df[['title', 'body']].fillna('').astype(str)

subset = df.head(75).copy()

predicted_lists = []
per_label_json  = []

for i, (_, row) in enumerate(subset.iterrows(), start=1):
    final_labels, per_label = predict_for_post(row['title'], row['body'])
    predicted_lists.append(final_labels)
    per_label_json.append(json.dumps(per_label, ensure_ascii=False))
    print(f"{i}. Labels: {final_labels}")

# Save results to CSV (labels as JSON strings for easy downstream parsing)
out = subset.copy()
out['PredictedLabels'] = [json.dumps(x, ensure_ascii=False) for x in predicted_lists]
out['PerLabelYESNO']   = per_label_json
out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"\nSaved predictions to {OUTPUT_CSV}")


🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
1. Labels: ['Medical', 'Mental Health', 'Pregnancy']
🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
🧠 Classifying post (Together AI - Llama)
✅ LLM response received.
🧠 Classifying post (Tog

In [10]:
# Cell 14 : Precision/Recall/F1 per post for prompt 1
# Cell: Precision/Recall/F1 per post using GT "Tags" column (Ubuntu-friendly)
import os
import ast
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# ---- paths (edit if your files live elsewhere) ----
INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"   # ground truth with a 'Tags' column
OUTPUT_CSV = "predicted_labels_75_0_shot_llama.csv"                      # predictions with 'PredictedLabels' column
SAVE_AS    = "per_post_metrics_from_tags_0_shot_llama.csv"                         # will be saved in the current folder

# ---- load data ----
gt_df   = pd.read_csv(INPUT_CSV)
pred_df = pd.read_csv(OUTPUT_CSV)

# align to first 75 rows (adjust if you want more)
gt_subset   = gt_df.head(75).reset_index(drop=True)
pred_subset = pred_df.head(75).reset_index(drop=True)

# ---- helpers ----
def parse_listish(value):
    """
    Parse labels that might be stored as python-list strings or comma-separated strings.
    Returns a set of normalized labels.
    """
    if pd.isna(value):
        return set()
    s = str(value).strip()
    # try python list literal first
    try:
        maybe_list = ast.literal_eval(s)
        if isinstance(maybe_list, (list, tuple)):
            items = maybe_list
        else:
            items = [s]
    except Exception:
        # fallback: comma-separated
        items = [tok.strip() for tok in s.split(",") if tok.strip()]
    # normalize labels (consistent spacing/case)
    return set([str(x).strip() for x in items if str(x).strip()])

# ---- parse labels ----
if "Tags" not in gt_subset.columns:
    raise KeyError("Ground truth CSV must contain a 'Tags' column with human-annotated labels.")

gt_labels_list   = gt_subset["Tags"].apply(parse_listish).tolist()
pred_labels_list = pred_subset["PredictedLabels"].apply(parse_listish).tolist()

# build label universe
all_labels = sorted(list(set().union(*gt_labels_list, *pred_labels_list)))

# ---- compute metrics ----
rows = []
y_true_all, y_pred_all = [], []

for idx, (gt_set, pred_set) in enumerate(zip(gt_labels_list, pred_labels_list), start=1):
    y_true = [1 if l in gt_set else 0 for l in all_labels]
    y_pred = [1 if l in pred_set else 0 for l in all_labels]

    y_true_all.extend(y_true)
    y_pred_all.extend(y_pred)

    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)

    rows.append({
        "PostIndex": idx,
        "TrueLabels": sorted(list(gt_set)),
        "PredictedLabels": sorted(list(pred_set)),
        "Precision": round(p, 3),
        "Recall": round(r, 3),
        "F1": round(f, 3),
    })

metrics_df = pd.DataFrame(rows)

# macro averages = mean of per-post metrics
macro_p = float(metrics_df["Precision"].mean())
macro_r = float(metrics_df["Recall"].mean())
macro_f = float(metrics_df["F1"].mean())

# micro averages = computed on pooled one-hot vectors
micro_p = precision_score(y_true_all, y_pred_all, zero_division=0)
micro_r = recall_score(y_true_all, y_pred_all, zero_division=0)
micro_f = f1_score(y_true_all, y_pred_all, zero_division=0)

summary_rows = pd.DataFrame([
    {"PostIndex": "Macro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(macro_p, 3), "Recall": round(macro_r, 3), "F1": round(macro_f, 3)},
    {"PostIndex": "Micro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(micro_p, 3), "Recall": round(micro_r, 3), "F1": round(micro_f, 3)},
])

final_metrics = pd.concat([metrics_df, summary_rows], ignore_index=True)

# ---- save (no /mnt/data, so always local) ----
final_metrics.to_csv(SAVE_AS, index=False, encoding="utf-8")
print(f"✅ Saved per-post metrics + macro/micro averages to: {os.path.abspath(SAVE_AS)}")


✅ Saved per-post metrics + macro/micro averages to: /home/ubuntu/per_post_metrics_from_tags_0_shot_llama.csv


In [11]:
# Multishot k - 0
#  Notebook cell: Summarize per-post metrics CSV (advisor-friendly printout)
import pandas as pd
import ast

# === Configure this to your file ===
CSV_PATH = "per_post_metrics_from_tags_0_shot_llama.csv"

def _safe_list(x):
    """Parse list-like strings into Python lists; return [] if parsing fails."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    # try JSON-ish or Python literal list
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return val
    except Exception:
        pass
    # fallback: comma-split
    return [t.strip() for t in s.split(",") if t.strip()]

def summarize_metrics(csv_path: str):
    df = pd.read_csv(csv_path)

    # Detect summary rows if present
    has_macro = (df["PostIndex"] == "Macro-Avg").any() if "PostIndex" in df.columns else False
    has_micro = (df["PostIndex"] == "Micro-Avg").any() if "PostIndex" in df.columns else False

    # Pull macro/micro if available, else compute from per-post rows
    macro_p = macro_r = macro_f = None
    micro_p = micro_r = micro_f = None

    if has_macro:
        row = df.loc[df["PostIndex"] == "Macro-Avg"].iloc[0]
        macro_p, macro_r, macro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])
    if has_micro:
        row = df.loc[df["PostIndex"] == "Micro-Avg"].iloc[0]
        micro_p, micro_r, micro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])

    # Filter per-post rows
    per_post = df.copy()
    if "PostIndex" in per_post.columns:
        per_post = per_post[~per_post["PostIndex"].isin(["Macro-Avg", "Micro-Avg"])]

    # Compute perfect/incorrect if we have labels
    perfect_matches = completely_incorrect = None
    if {"TrueLabels", "PredictedLabels"}.issubset(per_post.columns):
        t_labels = per_post["TrueLabels"].apply(_safe_list)
        p_labels = per_post["PredictedLabels"].apply(_safe_list)

        perfect_matches = sum(set(t) == set(p) for t, p in zip(t_labels, p_labels))
        completely_incorrect = sum(len(set(t).intersection(set(p))) == 0 for t, p in zip(t_labels, p_labels))
        n_posts = len(per_post)
    else:
        n_posts = len(per_post)

    # If macro/micro not present, approximate macro as mean of per-post, and skip micro
    if macro_p is None and {"Precision","Recall","F1"}.issubset(per_post.columns):
        macro_p = per_post["Precision"].mean()
        macro_r = per_post["Recall"].mean()
        macro_f = per_post["F1"].mean()

    # Pretty print in your requested format
    print("Here’s the summary you can share with your advisor:\n")
    if macro_p is not None:
        print(f"Macro Precision: {macro_p:.3f}")
        print(f"Macro Recall: {macro_r:.3f}")
        print(f"Macro F1: {macro_f:.3f}\n")
    else:
        print("Macro metrics: not available in this file.\n")

    if micro_p is not None:
        print(f"Micro Precision: {micro_p:.3f}")
        print(f"Micro Recall: {micro_r:.3f}")
        print(f"Micro F1: {micro_f:.3f}\n")
    else:
        print("Micro metrics: not available in this file.\n")

    if perfect_matches is not None:
        print(f"Perfect Matches: {perfect_matches} out of {n_posts} posts ({100*perfect_matches/n_posts:.1f}%) exactly matched human annotations")
    else:
        print("Perfect Matches: (labels not found in CSV to compute)")
    if completely_incorrect is not None:
        print(f"Completely Incorrect: {completely_incorrect} out of {n_posts} posts ({100*completely_incorrect/n_posts:.1f}%) had no correct labels")
    else:
        print("Completely Incorrect: (labels not found in CSV to compute)")

# Run
summarize_metrics(CSV_PATH)


Here’s the summary you can share with your advisor:

Macro Precision: 0.740
Macro Recall: 0.574
Macro F1: 0.610

Micro Precision: 0.790
Micro Recall: 0.524
Micro F1: 0.630

Perfect Matches: 24 out of 75 posts (32.0%) exactly matched human annotations
Completely Incorrect: 15 out of 75 posts (20.0%) had no correct labels


In [20]:
# # Cell 12 : Zero shot all 75 posts prompt 2
# # Cell: Batch 0-shot labeling for 75 posts (title+body) -> CSV
# import pandas as pd
# import time, re, json

# # Assumes these already exist in your environment:
# # - labels: List[str] (includes "Not Applicable" or not — both fine)
# # - definitions: Dict[str, str]
# # - make_per_label_prompt(title, body, label, definition)
# # - getTriggerWarningsLLMResponse(prompt)

# INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"
# OUTPUT_CSV = "predicted_labels_75_prompt_2_llama.csv"

# def normalize_yes_no(text: str) -> str:
#     """Parse any LLM response and return 'YES' or 'NO' (default NO)."""
#     if not text:
#         return "NO"
#     s = str(text).strip()
#     m = re.search(r"\b(YES|NO)\b", s, flags=re.I)
#     if m:
#         return m.group(1).upper()
#     # fallback: first char heuristic
#     return "YES" if s.upper().startswith("Y") else "NO"

# def predict_for_post(title: str, body: str):
#     """Run per-label 0-shot prompts and return final labels + per-label decisions."""
#     per_label = {}
#     for label in labels:
#         definition = definitions[label]
#         prompt = make_per_label_prompt(title, body, label, definition)
#         resp = getTriggerWarningsLLMResponse_Llama(prompt)
#         per_label[label] = normalize_yes_no(resp)
#         # tiny pause to be gentle with rate limits (tune/remove as needed)
#         time.sleep(0.05)

#     # Build final labels: if none are YES (excluding NA), mark Not Applicable
#     positives = [lbl for lbl, ans in per_label.items() if ans == "YES" and lbl != "Not Applicable"]
#     final_labels = positives if positives else ["Not Applicable"]
#     return final_labels, per_label

# # --- Load dataset (expects 'title' and 'body' columns) ---
# df = pd.read_csv(INPUT_CSV)
# df = df[['title', 'body']].fillna('').astype(str)

# subset = df.head(75).copy()

# predicted_lists = []
# per_label_json  = []

# for i, (_, row) in enumerate(subset.iterrows(), start=1):
#     final_labels, per_label = predict_for_post(row['title'], row['body'])
#     predicted_lists.append(final_labels)
#     per_label_json.append(json.dumps(per_label, ensure_ascii=False))
#     print(f"{i}. Labels: {final_labels}")

# # Save results to CSV (labels as JSON strings for easy downstream parsing)
# out = subset.copy()
# out['PredictedLabels'] = [json.dumps(x, ensure_ascii=False) for x in predicted_lists]
# out['PerLabelYESNO']   = per_label_json
# out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

# print(f"\nSaved predictions to {OUTPUT_CSV}")


In [21]:
# # Cell 14 : Precision/Recall/F1 per post for prompt 2
# # Cell: Precision/Recall/F1 per post using GT "Tags" column (Ubuntu-friendly)
# import os
# import ast
# import pandas as pd
# from sklearn.metrics import precision_score, recall_score, f1_score

# # ---- paths (edit if your files live elsewhere) ----
# INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"   # ground truth with a 'Tags' column
# OUTPUT_CSV = "predicted_labels_75_prompt_2_llama.csv"                      # predictions with 'PredictedLabels' column
# SAVE_AS    = "per_post_metrics_from_tags_Prompt_2_llama.csv"                         # will be saved in the current folder

# # ---- load data ----
# gt_df   = pd.read_csv(INPUT_CSV)
# pred_df = pd.read_csv(OUTPUT_CSV)

# # align to first 75 rows (adjust if you want more)
# gt_subset   = gt_df.head(75).reset_index(drop=True)
# pred_subset = pred_df.head(75).reset_index(drop=True)

# # ---- helpers ----
# def parse_listish(value):
#     """
#     Parse labels that might be stored as python-list strings or comma-separated strings.
#     Returns a set of normalized labels.
#     """
#     if pd.isna(value):
#         return set()
#     s = str(value).strip()
#     # try python list literal first
#     try:
#         maybe_list = ast.literal_eval(s)
#         if isinstance(maybe_list, (list, tuple)):
#             items = maybe_list
#         else:
#             items = [s]
#     except Exception:
#         # fallback: comma-separated
#         items = [tok.strip() for tok in s.split(",") if tok.strip()]
#     # normalize labels (consistent spacing/case)
#     return set([str(x).strip() for x in items if str(x).strip()])

# # ---- parse labels ----
# if "Tags" not in gt_subset.columns:
#     raise KeyError("Ground truth CSV must contain a 'Tags' column with human-annotated labels.")

# gt_labels_list   = gt_subset["Tags"].apply(parse_listish).tolist()
# pred_labels_list = pred_subset["PredictedLabels"].apply(parse_listish).tolist()

# # build label universe
# all_labels = sorted(list(set().union(*gt_labels_list, *pred_labels_list)))

# # ---- compute metrics ----
# rows = []
# y_true_all, y_pred_all = [], []

# for idx, (gt_set, pred_set) in enumerate(zip(gt_labels_list, pred_labels_list), start=1):
#     y_true = [1 if l in gt_set else 0 for l in all_labels]
#     y_pred = [1 if l in pred_set else 0 for l in all_labels]

#     y_true_all.extend(y_true)
#     y_pred_all.extend(y_pred)

#     p = precision_score(y_true, y_pred, zero_division=0)
#     r = recall_score(y_true, y_pred, zero_division=0)
#     f = f1_score(y_true, y_pred, zero_division=0)

#     rows.append({
#         "PostIndex": idx,
#         "TrueLabels": sorted(list(gt_set)),
#         "PredictedLabels": sorted(list(pred_set)),
#         "Precision": round(p, 3),
#         "Recall": round(r, 3),
#         "F1": round(f, 3),
#     })

# metrics_df = pd.DataFrame(rows)

# # macro averages = mean of per-post metrics
# macro_p = float(metrics_df["Precision"].mean())
# macro_r = float(metrics_df["Recall"].mean())
# macro_f = float(metrics_df["F1"].mean())

# # micro averages = computed on pooled one-hot vectors
# micro_p = precision_score(y_true_all, y_pred_all, zero_division=0)
# micro_r = recall_score(y_true_all, y_pred_all, zero_division=0)
# micro_f = f1_score(y_true_all, y_pred_all, zero_division=0)

# summary_rows = pd.DataFrame([
#     {"PostIndex": "Macro-Avg", "TrueLabels": None, "PredictedLabels": None,
#      "Precision": round(macro_p, 3), "Recall": round(macro_r, 3), "F1": round(macro_f, 3)},
#     {"PostIndex": "Micro-Avg", "TrueLabels": None, "PredictedLabels": None,
#      "Precision": round(micro_p, 3), "Recall": round(micro_r, 3), "F1": round(micro_f, 3)},
# ])

# final_metrics = pd.concat([metrics_df, summary_rows], ignore_index=True)

# # ---- save (no /mnt/data, so always local) ----
# final_metrics.to_csv(SAVE_AS, index=False, encoding="utf-8")
# print(f"✅ Saved per-post metrics + macro/micro averages to: {os.path.abspath(SAVE_AS)}")


### K-shot prompting for LLama

In [12]:
# Cell: Batch 1-shot (k=2) few-shot labeling for 75 posts -> CSV + used examples JSON
import pandas as pd
import json, time, copy, ast, re
import numpy as np

INPUT_CSV    = "Combined_Dataset_Annotations - Combined_Dataset.csv"
OUTPUT_CSV   = "predicted_labels_75_few_shot_k_2_llama.csv"
EXAMPLES_JSON = "used_examples_for_75_posts_K_2_llama.json"

# --- robust parsers / helpers ---
YESNO_RE = re.compile(r"\b(YES|NO)\b", re.IGNORECASE)

def extract_yes_no(resp) -> str:
    """
    Robustly coerce model output to YES/NO.
    - Accept dicts with .get('text'), .get('content'), .get('choices'[0]['text'])
    - Take only the first line, then first YES/NO token anywhere on that line.
    """
    if resp is None:
        return "NO"
    # unwrap common response shapes
    if isinstance(resp, dict):
        if "text" in resp and isinstance(resp["text"], str):
            resp = resp["text"]
        elif "content" in resp and isinstance(resp["content"], str):
            resp = resp["content"]
        elif "choices" in resp and isinstance(resp["choices"], list) and resp["choices"]:
            ch = resp["choices"][0]
            if isinstance(ch, dict):
                if "text" in ch:
                    resp = ch["text"]
                elif "message" in ch and isinstance(ch["message"], dict) and "content" in ch["message"]:
                    resp = ch["message"]["content"]
                else:
                    resp = str(ch)
            else:
                resp = str(ch)
        else:
            resp = str(resp)
    # must be string now
    if not isinstance(resp, str):
        resp = str(resp)

    line = resp.strip().splitlines()[0] if resp.strip() else ""
    m = YESNO_RE.search(line)
    if m:
        return "YES" if m.group(1).upper() == "YES" else "NO"

    # fallback: first token check
    tok = line.upper().split()[:1]
    return "YES" if (tok and tok[0] == "YES") else "NO"

def convert_to_serializable(obj):
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return obj

def tiny_throttle(sec=0.25):
    try:
        time.sleep(sec)
    except Exception:
        pass

# --- load input posts ---
df_in = pd.read_csv(INPUT_CSV)
if not {'title','body'}.issubset(df_in.columns):
    raise KeyError("INPUT_CSV must contain 'title' and 'body' columns.")

subset = df_in.head(75).copy()
subset = subset[['title','body']].fillna('').astype(str)

predicted_lists = []
all_used_examples = []   # snapshot of examples used per post

for i, (_, row) in enumerate(subset.iterrows(), start=1):
    title = row['title']
    body  = row['body']

    # Warm up cosine neighbors (populates collected_examples_per_label)
    _ = make_multishot_prompt_cosine(
        k=1,
        title=title,
        body=body,
        current_index=None,
        include_defs=True,
        debug=False
    )

    positive_labels = []

    for j, lbl in enumerate(labels):
        single_prompt = make_prompt_for_label(
            label_name=lbl,
            k=1,
            title=title,
            body=body,
            current_index=None,
            debug=False
        )

        # First attempt (tight)
        resp = getTriggerWarningsLLMResponse_Llama(
            single_prompt,
            mode="raw",
            max_tokens=4,
            temperature=0.0,
            verbose=False
        )
        ans = extract_yes_no(resp)

        # If we didn't get a clean YES/NO on first line, retry with a slightly bigger budget
        if ans not in ("YES", "NO"):
            resp = getTriggerWarningsLLMResponse_Llama(
                single_prompt,
                mode="raw",
                max_tokens=8,   # allow newline + token quirks
                temperature=0.0,
                verbose=False
            )
            ans = extract_yes_no(resp)

        # Light logging for first 2 posts × first 3 labels to diagnose
        if i <= 2 and j < 3:
            print("\n--- DEBUG RAW ---")
            print(f"Post {i}, Label '{lbl}':")
            # Show a compact view
            if isinstance(resp, str):
                print(resp[:200].replace("\n", "\\n"))
            else:
                s = json.dumps(resp, default=str) if not isinstance(resp, str) else resp
                print(s[:200].replace("\n", "\\n"))
            print(f"Parsed: {ans}")
            print("-----------------")

        if ans == "YES":
            positive_labels.append(lbl)

        tiny_throttle(0.15)

    if not positive_labels:
        positive_labels = ["Not Applicable"]

    predicted_lists.append(positive_labels)

    snapshot = copy.deepcopy(collected_examples_per_label) if 'collected_examples_per_label' in globals() else {}
    all_used_examples.append({
        "post_index_1_based": i,
        "title": title,
        "body": body,
        "examples_by_label": snapshot
    })

    print(f"{i}. Labels: {positive_labels}")

# --- save predictions CSV ---
out = subset.copy()
out['PredictedLabels'] = [json.dumps(x, ensure_ascii=False) for x in predicted_lists]
out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"\n✅ Saved predictions to {OUTPUT_CSV}")

# --- save used examples JSON ---
with open(EXAMPLES_JSON, "w", encoding="utf-8") as f:
    json.dump(all_used_examples, f, indent=2, ensure_ascii=False, default=convert_to_serializable)
print(f"✅ Saved used examples to {EXAMPLES_JSON}")


In [19]:
# # Cell: Batch 1-shot (k=2) few-shot labeling for 75 posts -> CSV + used examples JSON
# import pandas as pd
# import json, time, copy, ast, re
# import numpy as np

# INPUT_CSV    = "Combined_Dataset_Annotations - Combined_Dataset.csv"
# OUTPUT_CSV   = "predicted_labels_75_few_shot_k_2_llama.csv"
# EXAMPLES_JSON = "used_examples_for_75_posts_K_2_llama.json"

# # --- robust parsers / helpers ---
# YESNO_RE = re.compile(r"\b(YES|NO)\b", re.IGNORECASE)

# def extract_yes_no(resp) -> str:
#     """
#     Robustly coerce model output to YES/NO.
#     - Accept dicts with .get('text'), .get('content'), .get('choices'[0]['text'])
#     - Take only the first line, then first YES/NO token anywhere on that line.
#     """
#     if resp is None:
#         return "NO"
#     # unwrap common response shapes
#     if isinstance(resp, dict):
#         if "text" in resp and isinstance(resp["text"], str):
#             resp = resp["text"]
#         elif "content" in resp and isinstance(resp["content"], str):
#             resp = resp["content"]
#         elif "choices" in resp and isinstance(resp["choices"], list) and resp["choices"]:
#             ch = resp["choices"][0]
#             if isinstance(ch, dict):
#                 if "text" in ch:
#                     resp = ch["text"]
#                 elif "message" in ch and isinstance(ch["message"], dict) and "content" in ch["message"]:
#                     resp = ch["message"]["content"]
#                 else:
#                     resp = str(ch)
#             else:
#                 resp = str(ch)
#         else:
#             resp = str(resp)
#     # must be string now
#     if not isinstance(resp, str):
#         resp = str(resp)

#     line = resp.strip().splitlines()[0] if resp.strip() else ""
#     m = YESNO_RE.search(line)
#     if m:
#         return "YES" if m.group(1).upper() == "YES" else "NO"

#     # fallback: first token check
#     tok = line.upper().split()[:1]
#     return "YES" if (tok and tok[0] == "YES") else "NO"

# def convert_to_serializable(obj):
#     if isinstance(obj, (np.float32, np.float64)):
#         return float(obj)
#     if isinstance(obj, (np.int32, np.int64)):
#         return int(obj)
#     return obj

# def tiny_throttle(sec=0.25):
#     try:
#         time.sleep(sec)
#     except Exception:
#         pass

# # --- load input posts ---
# df_in = pd.read_csv(INPUT_CSV)
# if not {'title','body'}.issubset(df_in.columns):
#     raise KeyError("INPUT_CSV must contain 'title' and 'body' columns.")

# subset = df_in.head(75).copy()
# subset = subset[['title','body']].fillna('').astype(str)

# predicted_lists = []
# all_used_examples = []   # snapshot of examples used per post

# for i, (_, row) in enumerate(subset.iterrows(), start=1):
#     title = row['title']
#     body  = row['body']

#     # Warm up cosine neighbors (populates collected_examples_per_label)
#     _ = make_multishot_prompt_cosine(
#         k=2,
#         title=title,
#         body=body,
#         current_index=None,
#         include_defs=True,
#         debug=False
#     )

#     positive_labels = []

#     for j, lbl in enumerate(labels):
#         single_prompt = make_prompt_for_label(
#             label_name=lbl,
#             k=2,
#             title=title,
#             body=body,
#             current_index=None,
#             debug=False
#         )

#         # First attempt (tight)
#         resp = getTriggerWarningsLLMResponse_Llama(
#             single_prompt,
#             mode="raw",
#             max_tokens=4,
#             temperature=0.0,
#             verbose=False
#         )
#         ans = extract_yes_no(resp)

#         # If we didn't get a clean YES/NO on first line, retry with a slightly bigger budget
#         if ans not in ("YES", "NO"):
#             resp = getTriggerWarningsLLMResponse_Llama(
#                 single_prompt,
#                 mode="raw",
#                 max_tokens=8,   # allow newline + token quirks
#                 temperature=0.0,
#                 verbose=False
#             )
#             ans = extract_yes_no(resp)

#         # Light logging for first 2 posts × first 3 labels to diagnose
#         if i <= 2 and j < 3:
#             print("\n--- DEBUG RAW ---")
#             print(f"Post {i}, Label '{lbl}':")
#             # Show a compact view
#             if isinstance(resp, str):
#                 print(resp[:200].replace("\n", "\\n"))
#             else:
#                 s = json.dumps(resp, default=str) if not isinstance(resp, str) else resp
#                 print(s[:200].replace("\n", "\\n"))
#             print(f"Parsed: {ans}")
#             print("-----------------")

#         if ans == "YES":
#             positive_labels.append(lbl)

#         tiny_throttle(0.15)

#     # --- Not Applicable exclusivity + fallback ---
#     # If NA shows up with others, force it to be standalone
#     if "Not Applicable" in positive_labels and len(positive_labels) > 1:
#         positive_labels = ["Not Applicable"]

#     # If nothing matched at all, default to NA
#     if not positive_labels:
#         positive_labels = ["Not Applicable"]

#     predicted_lists.append(positive_labels)

#     snapshot = copy.deepcopy(collected_examples_per_label) if 'collected_examples_per_label' in globals() else {}
#     all_used_examples.append({
#         "post_index_1_based": i,
#         "title": title,
#         "body": body,
#         "examples_by_label": snapshot
#     })

#     print(f"{i}. Labels: {positive_labels}")

# # --- save predictions CSV ---
# out = subset.copy()
# out['PredictedLabels'] = [json.dumps(x, ensure_ascii=False) for x in predicted_lists]
# out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
# print(f"\n✅ Saved predictions to {OUTPUT_CSV}")

# # --- save used examples JSON ---
# with open(EXAMPLES_JSON, "w", encoding="utf-8") as f:
#     json.dump(all_used_examples, f, indent=2, ensure_ascii=False, default=convert_to_serializable)
# print(f"✅ Saved used examples to {EXAMPLES_JSON}")


In [17]:
# Cell: Precision/Recall/F1 per post using GT "Tags" column (Ubuntu-friendly)
import os
import ast
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# ---- paths (edit if your files live elsewhere) ----
INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"   # ground truth with a 'Tags' column
OUTPUT_CSV = "predicted_labels_75_few_shot_k_2_llama.csv"                      # predictions with 'PredictedLabels' column
SAVE_AS    = "per_post_metrics_from_tags_multishot_k_2_llama.csv"                         # will be saved in the current folder

# ---- load data ----
gt_df   = pd.read_csv(INPUT_CSV)
pred_df = pd.read_csv(OUTPUT_CSV)

# align to first 75 rows (adjust if you want more)
gt_subset   = gt_df.head(75).reset_index(drop=True)
pred_subset = pred_df.head(75).reset_index(drop=True)

# ---- helpers ----
def parse_listish(value):
    """
    Parse labels that might be stored as python-list strings or comma-separated strings.
    Returns a set of normalized labels.
    """
    if pd.isna(value):
        return set()
    s = str(value).strip()
    # try python list literal first
    try:
        maybe_list = ast.literal_eval(s)
        if isinstance(maybe_list, (list, tuple)):
            items = maybe_list
        else:
            items = [s]
    except Exception:
        # fallback: comma-separated
        items = [tok.strip() for tok in s.split(",") if tok.strip()]
    # normalize labels (consistent spacing/case)
    return set([str(x).strip() for x in items if str(x).strip()])

# ---- parse labels ----
if "Tags" not in gt_subset.columns:
    raise KeyError("Ground truth CSV must contain a 'Tags' column with human-annotated labels.")

gt_labels_list   = gt_subset["Tags"].apply(parse_listish).tolist()
pred_labels_list = pred_subset["PredictedLabels"].apply(parse_listish).tolist()

# build label universe
all_labels = sorted(list(set().union(*gt_labels_list, *pred_labels_list)))

# ---- compute metrics ----
rows = []
y_true_all, y_pred_all = [], []

for idx, (gt_set, pred_set) in enumerate(zip(gt_labels_list, pred_labels_list), start=1):
    y_true = [1 if l in gt_set else 0 for l in all_labels]
    y_pred = [1 if l in pred_set else 0 for l in all_labels]

    y_true_all.extend(y_true)
    y_pred_all.extend(y_pred)

    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)

    rows.append({
        "PostIndex": idx,
        "TrueLabels": sorted(list(gt_set)),
        "PredictedLabels": sorted(list(pred_set)),
        "Precision": round(p, 3),
        "Recall": round(r, 3),
        "F1": round(f, 3),
    })

metrics_df = pd.DataFrame(rows)

# macro averages = mean of per-post metrics
macro_p = float(metrics_df["Precision"].mean())
macro_r = float(metrics_df["Recall"].mean())
macro_f = float(metrics_df["F1"].mean())

# micro averages = computed on pooled one-hot vectors
micro_p = precision_score(y_true_all, y_pred_all, zero_division=0)
micro_r = recall_score(y_true_all, y_pred_all, zero_division=0)
micro_f = f1_score(y_true_all, y_pred_all, zero_division=0)

summary_rows = pd.DataFrame([
    {"PostIndex": "Macro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(macro_p, 3), "Recall": round(macro_r, 3), "F1": round(macro_f, 3)},
    {"PostIndex": "Micro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(micro_p, 3), "Recall": round(micro_r, 3), "F1": round(micro_f, 3)},
])

final_metrics = pd.concat([metrics_df, summary_rows], ignore_index=True)

# ---- save (no /mnt/data, so always local) ----
final_metrics.to_csv(SAVE_AS, index=False, encoding="utf-8")
print(f"✅ Saved per-post metrics + macro/micro averages to: {os.path.abspath(SAVE_AS)}")


✅ Saved per-post metrics + macro/micro averages to: /home/ubuntu/per_post_metrics_from_tags_multishot_k_2_llama.csv


In [18]:
# Multishot k - 1
#  Notebook cell: Summarize per-post metrics CSV (advisor-friendly printout)
import pandas as pd
import ast

# === Configure this to your file ===
CSV_PATH = "per_post_metrics_from_tags_multishot_k_2_llama.csv"

def _safe_list(x):
    """Parse list-like strings into Python lists; return [] if parsing fails."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    # try JSON-ish or Python literal list
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return val
    except Exception:
        pass
    # fallback: comma-split
    return [t.strip() for t in s.split(",") if t.strip()]

def summarize_metrics(csv_path: str):
    df = pd.read_csv(csv_path)

    # Detect summary rows if present
    has_macro = (df["PostIndex"] == "Macro-Avg").any() if "PostIndex" in df.columns else False
    has_micro = (df["PostIndex"] == "Micro-Avg").any() if "PostIndex" in df.columns else False

    # Pull macro/micro if available, else compute from per-post rows
    macro_p = macro_r = macro_f = None
    micro_p = micro_r = micro_f = None

    if has_macro:
        row = df.loc[df["PostIndex"] == "Macro-Avg"].iloc[0]
        macro_p, macro_r, macro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])
    if has_micro:
        row = df.loc[df["PostIndex"] == "Micro-Avg"].iloc[0]
        micro_p, micro_r, micro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])

    # Filter per-post rows
    per_post = df.copy()
    if "PostIndex" in per_post.columns:
        per_post = per_post[~per_post["PostIndex"].isin(["Macro-Avg", "Micro-Avg"])]

    # Compute perfect/incorrect if we have labels
    perfect_matches = completely_incorrect = None
    if {"TrueLabels", "PredictedLabels"}.issubset(per_post.columns):
        t_labels = per_post["TrueLabels"].apply(_safe_list)
        p_labels = per_post["PredictedLabels"].apply(_safe_list)

        perfect_matches = sum(set(t) == set(p) for t, p in zip(t_labels, p_labels))
        completely_incorrect = sum(len(set(t).intersection(set(p))) == 0 for t, p in zip(t_labels, p_labels))
        n_posts = len(per_post)
    else:
        n_posts = len(per_post)

    # If macro/micro not present, approximate macro as mean of per-post, and skip micro
    if macro_p is None and {"Precision","Recall","F1"}.issubset(per_post.columns):
        macro_p = per_post["Precision"].mean()
        macro_r = per_post["Recall"].mean()
        macro_f = per_post["F1"].mean()

    # Pretty print in your requested format
    print("Here’s the summary you can share with your advisor:\n")
    if macro_p is not None:
        print(f"Macro Precision: {macro_p:.3f}")
        print(f"Macro Recall: {macro_r:.3f}")
        print(f"Macro F1: {macro_f:.3f}\n")
    else:
        print("Macro metrics: not available in this file.\n")

    if micro_p is not None:
        print(f"Micro Precision: {micro_p:.3f}")
        print(f"Micro Recall: {micro_r:.3f}")
        print(f"Micro F1: {micro_f:.3f}\n")
    else:
        print("Micro metrics: not available in this file.\n")

    if perfect_matches is not None:
        print(f"Perfect Matches: {perfect_matches} out of {n_posts} posts ({100*perfect_matches/n_posts:.1f}%) exactly matched human annotations")
    else:
        print("Perfect Matches: (labels not found in CSV to compute)")
    if completely_incorrect is not None:
        print(f"Completely Incorrect: {completely_incorrect} out of {n_posts} posts ({100*completely_incorrect/n_posts:.1f}%) had no correct labels")
    else:
        print("Completely Incorrect: (labels not found in CSV to compute)")

# Run
summarize_metrics(CSV_PATH)


Here’s the summary you can share with your advisor:

Macro Precision: 0.539
Macro Recall: 0.452
Macro F1: 0.458

Micro Precision: 0.630
Micro Recall: 0.401
Micro F1: 0.490

Perfect Matches: 13 out of 75 posts (17.3%) exactly matched human annotations
Completely Incorrect: 29 out of 75 posts (38.7%) had no correct labels
